# EP08. JSON Mode & Function Calling
## LLM에게 깔끔한 결과값을 받아내는 컨텍스트 설계법

---

### 학습 목표

1. **Pydantic + LangChain `.with_structured_output()`** — 타입 안전한 구조화 출력 구현
2. **instructor로 구조화 출력** — Claude/OpenAI 양쪽에서 동일한 인터페이스 사용
3. **파싱 실패 자동 재시도** — tenacity @retry와 OutputFixingParser 구현
4. **Langfuse 성공률 추적** — 파싱 성공/실패 스코어 기록 및 대시보드 모니터링

---

### AI 가이드 안내

> 이 노트북은 Claude Code 또는 ChatGPT와 함께 학습하도록 설계되어 있습니다.  
> 각 셀 하단의 `# TODO` 주석이 있는 부분은 직접 구현해 보세요.  
> 막히면 AI에게 "이 셀의 TODO를 구현해줘" 라고 질문하면 됩니다.

**권장 질문 예시**
- "`with_structured_output`과 `instructor`의 내부 동작 차이를 설명해줘"
- "Pydantic에서 `Field(description=...)` 을 잘 작성하는 팁을 알려줘"
- "tenacity의 `@retry` 데코레이터 사용법을 예시와 함께 설명해줘"

---
## 섹션 1. 환경 설정

필요한 패키지를 설치합니다.

In [ ]:
# uv pip install (권장)
# !uv pip install langchain langchain-anthropic langchain-openai langfuse instructor pydantic python-dotenv tenacity

# pip 사용 시
!pip install -q langchain langchain-anthropic langchain-openai langfuse instructor pydantic python-dotenv tenacity

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# 필수 환경 변수 확인
required_keys = ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
for key in required_keys:
    val = os.getenv(key)
    status = "OK" if val else "MISSING"
    print(f"{key}: {status}")

In [ ]:
# LangChain 임포트
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain_core.prompts import ChatPromptTemplate

# Pydantic
from pydantic import BaseModel, Field, field_validator
from typing import Optional, List, Literal, Union

# instructor
import instructor
import anthropic
from openai import OpenAI

# 재시도
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Langfuse
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

# 기타
import json
import time
import pandas as pd

# LLM 초기화
claude_llm = ChatAnthropic(model="claude-3-5-haiku-20241022", temperature=0)
gpt_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("라이브러리 로드 완료")

---
## 섹션 3. Pydantic 모델 정의

뉴스 기사, 상품 리뷰, 이력서 3가지 스키마를 정의합니다.

In [ ]:
# ===== 스키마 1: 뉴스 기사 =====
class NewsArticle(BaseModel):
    """뉴스 기사 구조화 스키마. 기사 텍스트에서 핵심 정보를 추출합니다."""

    title: str = Field(description="기사 제목 (100자 이내로 요약)")
    category: Literal["정치", "경제", "사회", "문화", "IT", "스포츠", "국제"] = Field(
        description="기사 카테고리. 가장 적합한 카테고리 하나 선택"
    )
    summary: str = Field(description="기사 핵심 내용 요약 (3문장 이내)")
    keywords: List[str] = Field(description="핵심 키워드 3~5개")
    sentiment: Literal["긍정", "중립", "부정"] = Field(
        description="기사 전체 논조. 긍정적=좋은 소식, 부정적=나쁜 소식/문제, 중립=사실 전달"
    )
    published_date: Optional[str] = Field(
        default=None,
        description="발행일 (YYYY-MM-DD 형식, 기사에 명시된 경우에만)"
    )
    sources_mentioned: List[str] = Field(
        default_factory=list,
        description="기사에서 인용된 인물, 기관, 단체 이름 목록"
    )

print("NewsArticle 스키마:")
print(json.dumps(NewsArticle.model_json_schema(), indent=2, ensure_ascii=False)[:500] + "...")

In [ ]:
# ===== 스키마 2: 상품 리뷰 =====
class ProductReview(BaseModel):
    """상품 리뷰 구조화 스키마. 리뷰 텍스트에서 평가 정보를 추출합니다."""

    product_name: str = Field(description="리뷰 대상 상품명")
    rating: float = Field(
        ge=1.0, le=5.0,
        description="별점 (1.0~5.0). 명시적 별점이 없으면 텍스트 톤에서 추정"
    )
    pros: List[str] = Field(
        description="상품의 장점 목록 (각 항목 20자 이내)",
        min_length=1
    )
    cons: List[str] = Field(
        description="상품의 단점 목록 (각 항목 20자 이내). 없으면 빈 리스트",
        default_factory=list
    )
    recommend: bool = Field(
        description="구매 추천 여부. rating >= 4.0이면 보통 True"
    )
    sentiment_score: float = Field(
        ge=0.0, le=1.0,
        description="감성 점수 (0.0=매우 부정, 1.0=매우 긍정)"
    )
    key_features_mentioned: List[str] = Field(
        default_factory=list,
        description="언급된 제품 특징 (예: 배터리, 화질, 가격, 디자인)"
    )

print("ProductReview 스키마 정의 완료")
print(f"필드 수: {len(ProductReview.model_fields)}")

In [ ]:
# ===== 스키마 3: 이력서 (중첩 모델) =====
class WorkExperience(BaseModel):
    """직무 경험 항목"""
    company: str = Field(description="회사명")
    role: str = Field(description="직책/역할")
    years: float = Field(
        ge=0.0,
        description="근무 기간 (년, 예: 2.5 = 2년 6개월)"
    )
    skills: List[str] = Field(description="해당 직무에서 사용한 기술 스택")
    is_current: bool = Field(
        default=False,
        description="현재 재직 중 여부"
    )

class Education(BaseModel):
    """학력 항목"""
    school: str = Field(description="학교명")
    degree: Literal["학사", "석사", "박사", "전문학사", "기타"] = Field(
        description="학위 구분"
    )
    major: str = Field(description="전공")
    graduation_year: Optional[int] = Field(
        default=None,
        description="졸업 연도 (YYYY, 재학 중이면 None)"
    )

class Resume(BaseModel):
    """이력서 전체 구조. 이력서 텍스트에서 모든 정보를 추출합니다."""
    name: str = Field(description="지원자 이름")
    email: Optional[str] = Field(default=None, description="이메일 주소")
    phone: Optional[str] = Field(default=None, description="연락처")
    total_experience_years: float = Field(
        ge=0.0,
        description="총 경력 년수. experiences의 years 합산"
    )
    experiences: List[WorkExperience] = Field(
        description="직무 경험 목록 (최신순)"
    )
    education: List[Education] = Field(
        description="학력 목록 (최신순)"
    )
    top_skills: List[str] = Field(
        description="핵심 기술 스택 상위 5개 (빈도 + 최근성 기준)",
        max_length=5
    )
    languages: List[str] = Field(
        default_factory=list,
        description="사용 가능한 언어 (예: 한국어, 영어)"
    )

print("모든 스키마 정의 완료")
print(f"Resume 필드 수: {len(Resume.model_fields)}")
print(f"WorkExperience 필드 수: {len(WorkExperience.model_fields)}")

---
## 섹션 4. LangChain .with_structured_output() 기본 사용법

In [ ]:
# 샘플 뉴스 기사 텍스트
SAMPLE_NEWS = """
삼성전자, 3분기 영업이익 전년比 274% 급증...반도체 회복세 뚜렷

삼성전자가 올해 3분기 연결 기준 영업이익 9조1000억원을 기록했다고 8일 발표했다.
이는 전년 동기 대비 274% 증가한 수치로, 반도체 업황 회복의 신호탄이라는 분석이 나온다.
HBM(고대역폭메모리) 수요 증가와 서버용 D램 가격 상승이 주요 원인으로 꼽힌다.
삼성전자 반도체(DS) 부문은 6조원대의 영업이익을 기록하며 흑자 전환에 성공했다.
증권가에서는 4분기에도 실적 개선세가 이어질 것으로 전망하고 있다.
하나증권 김현수 애널리스트는 "AI 서버 투자 확대로 HBM 수요가 2025년까지 급증할 것"이라고 말했다.
"""

# .with_structured_output() 사용
structured_claude = claude_llm.with_structured_output(NewsArticle)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 분석 전문가입니다. 주어진 기사에서 정보를 정확하게 추출하세요."),
    ("human", "다음 기사를 분석하여 구조화된 데이터로 추출하세요:\n\n{article}")
])

chain = prompt_template | structured_claude
result = chain.invoke({"article": SAMPLE_NEWS})

print(f"타입: {type(result)}")
print(f"제목: {result.title}")
print(f"카테고리: {result.category}")
print(f"감성: {result.sentiment}")
print(f"키워드: {result.keywords}")
print(f"요약: {result.summary}")
print(f"언급 출처: {result.sources_mentioned}")

---
## 섹션 5. 복잡한 중첩 Pydantic 모델

`Resume` 스키마로 중첩된 구조를 파싱합니다.

In [ ]:
SAMPLE_RESUME = """
김도현 (Kim Dohyun)
이메일: dohyun.kim@example.com | 전화: 010-1234-5678

경력
──────────────────────────────────────
카카오 (2022.03 ~ 현재)
시니어 ML 엔지니어
- 추천 시스템 MLOps 파이프라인 구축 (Kubeflow, MLflow)
- LLM 기반 대화형 검색 시스템 개발 (LangChain, RAG)
- A/B 테스트 플랫폼 설계 및 운영
주요 기술: Python, PyTorch, Kubernetes, LangChain, Spark

네이버 (2019.01 ~ 2022.02)
ML 엔지니어
- 검색 랭킹 모델 개발 및 배포 (3년)
- 실시간 추천 API 개발 (FastAPI, Redis)
주요 기술: Python, TensorFlow, FastAPI, Redis, Docker

학력
──────────────────────────────────────
서울대학교 컴퓨터공학과 석사 (2017~2019)
연세대학교 전자공학과 학사 (2013~2017)

언어: 한국어 (모국어), 영어 (업무 가능)
"""

structured_gpt = gpt_llm.with_structured_output(Resume)

resume_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 이력서 파싱 전문가입니다. 이력서에서 모든 정보를 정확히 추출하세요."),
    ("human", "다음 이력서를 파싱하세요:\n\n{resume}")
])

resume_chain = resume_prompt | structured_gpt
parsed_resume = resume_chain.invoke({"resume": SAMPLE_RESUME})

print(f"이름: {parsed_resume.name}")
print(f"이메일: {parsed_resume.email}")
print(f"총 경력: {parsed_resume.total_experience_years}년")
print(f"핵심 기술: {parsed_resume.top_skills}")
print(f"\n직무 경험 ({len(parsed_resume.experiences)}개):")
for exp in parsed_resume.experiences:
    print(f"  - {exp.company} | {exp.role} | {exp.years}년 | 현재: {exp.is_current}")
print(f"\n학력 ({len(parsed_resume.education)}개):")
for edu in parsed_resume.education:
    print(f"  - {edu.school} | {edu.degree} | {edu.major} | {edu.graduation_year}")

---
## 섹션 6. instructor로 Claude 구조화 출력

In [ ]:
# instructor로 Anthropic 클라이언트 패치
anthropic_client = instructor.from_anthropic(
    anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
)

SAMPLE_REVIEW = """
갤럭시 S24 울트라 구매한 지 3개월 됐는데 정말 만족스럽습니다.
S펜이 생각보다 훨씬 유용하고, 특히 노트 필기할 때 압도적입니다.
카메라 줌이 200배까지 되는데 실제로 20~30배 정도가 제일 실용적이에요.
배터리도 하루 종일 버텨주고 충전 속도도 빠릅니다.
단점이라면 무게가 좀 있고, 가격이 비싸다는 점이요.
하지만 전반적으로 최고의 안드로이드폰이라 생각합니다. 강력 추천!
"""

# instructor로 구조화 출력
review_result = anthropic_client.messages.create(
    model="claude-3-5-haiku-20241022",
    max_tokens=1024,
    response_model=ProductReview,
    messages=[
        {
            "role": "user",
            "content": f"다음 제품 리뷰를 분석하세요:\n\n{SAMPLE_REVIEW}"
        }
    ]
)

print(f"타입: {type(review_result)}")
print(f"상품명: {review_result.product_name}")
print(f"별점: {review_result.rating}/5.0")
print(f"장점: {review_result.pros}")
print(f"단점: {review_result.cons}")
print(f"추천: {review_result.recommend}")
print(f"감성 점수: {review_result.sentiment_score}")
print(f"언급 특징: {review_result.key_features_mentioned}")

---
## 섹션 7. instructor로 OpenAI 구조화 출력 비교

In [ ]:
# instructor로 OpenAI 클라이언트 패치
openai_client = instructor.from_openai(
    OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
)

# 동일한 리뷰로 OpenAI 테스트
openai_review = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=ProductReview,
    messages=[
        {
            "role": "user",
            "content": f"다음 제품 리뷰를 분석하세요:\n\n{SAMPLE_REVIEW}"
        }
    ]
)

# 두 결과 비교
print("=== Claude vs GPT-4o-mini 비교 ===")
comparison_data = {
    "필드": ["product_name", "rating", "recommend", "sentiment_score", "pros 수", "cons 수"],
    "Claude Haiku": [
        review_result.product_name,
        review_result.rating,
        review_result.recommend,
        review_result.sentiment_score,
        len(review_result.pros),
        len(review_result.cons)
    ],
    "GPT-4o-mini": [
        openai_review.product_name,
        openai_review.rating,
        openai_review.recommend,
        openai_review.sentiment_score,
        len(openai_review.pros),
        len(openai_review.cons)
    ]
}
df_compare = pd.DataFrame(comparison_data)
print(df_compare.to_string(index=False))

---
## 섹션 8. 파싱 실패 시나리오 시뮬레이션

의도적으로 파싱하기 어려운 입력을 만들어 실패 케이스를 관찰합니다.

In [ ]:
# 파싱하기 어려운 입력들
tricky_inputs = [
    # 1. 아이러니 표현 (실제 감성과 표현이 반대)
    "와 진짜 최고네요 ㅋㅋㅋ 배터리 2시간 만에 방전되고 카메라는 흐릿하고 가격은 100만원... 완전 추천합니다^^;",
    # 2. 매우 짧고 정보 없는 리뷰
    "그냥 그래요.",
    # 3. 영어/한국어 혼용
    "This product is 완전 별로. Battery life는 terrible하고 price도 너무 비싸. Not recommended at all 솔직히.",
    # 4. 별점 범위 외 표현
    "10점 만점에 6점 드립니다. 그럭저럭 쓸만해요.",
    # 5. 상품명이 불명확한 경우
    "지난달에 산 거 쓰는데 생각보다 괜찮네요. 색상이 예쁘고 가격 대비 만족합니다."
]

def parse_review_basic(text: str, llm) -> dict:
    """기본 파싱 (재시도 없음)"""
    try:
        structured = llm.with_structured_output(ProductReview)
        result = structured.invoke(f"다음 리뷰를 분석하세요:\n{text}")
        return {"success": True, "result": result, "error": None}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

print("=== 기본 파싱 결과 ===")
for i, text in enumerate(tricky_inputs, 1):
    outcome = parse_review_basic(text, gpt_llm)
    status = "성공" if outcome["success"] else "실패"
    if outcome["success"]:
        print(f"[{i}] {status}: rating={outcome['result'].rating}, recommend={outcome['result'].recommend}")
    else:
        print(f"[{i}] {status}: {outcome['error'][:80]}")

---
## 섹션 9. tenacity @retry로 자동 재시도 구현

In [ ]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
    RetryError
)
import logging

logger = logging.getLogger(__name__)

# 재시도 설정
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=5),
    retry=retry_if_exception_type(Exception),
    reraise=True
)
def parse_with_retry(text: str, llm, schema) -> BaseModel:
    """재시도 로직이 포함된 파싱 함수."""
    structured = llm.with_structured_output(schema)
    return structured.invoke(f"다음 내용을 분석하세요:\n{text}")


def safe_parse_with_retry(text: str, llm, schema) -> dict:
    """재시도 파싱 + 결과 래퍼."""
    try:
        result = parse_with_retry(text, llm, schema)
        return {"success": True, "result": result, "error": None}
    except RetryError as e:
        return {"success": False, "result": None, "error": f"RetryError: {e}"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}


print("=== 재시도 적용 파싱 결과 ===")
retry_success = 0
for i, text in enumerate(tricky_inputs, 1):
    outcome = safe_parse_with_retry(text, gpt_llm, ProductReview)
    status = "성공" if outcome["success"] else "실패"
    if outcome["success"]:
        retry_success += 1
        print(f"[{i}] {status}: rating={outcome['result'].rating:.1f}, sentiment={outcome['result'].sentiment_score:.2f}")
    else:
        print(f"[{i}] {status}: {outcome['error'][:80]}")

print(f"\n재시도 성공률: {retry_success}/{len(tricky_inputs)} ({retry_success/len(tricky_inputs):.0%})")

---
## 섹션 10. LangChain OutputFixingParser

In [ ]:
from langchain.output_parsers import PydanticOutputParser, OutputFixingParser

# 기본 파서
base_parser = PydanticOutputParser(pydantic_object=ProductReview)

# OutputFixingParser — 실패 시 LLM에게 수정 요청
fixing_parser = OutputFixingParser.from_llm(
    parser=base_parser,
    llm=gpt_llm
)

# 의도적으로 잘못된 JSON 예시들
malformed_outputs = [
    # 큰따옴표 없는 JSON
    "{product_name: 'iPhone', rating: 4.5, pros: ['좋음'], cons: [], recommend: true, sentiment_score: 0.8, key_features_mentioned: []}",
    # 설명 텍스트가 섞인 JSON
    "물론이죠! 분석 결과입니다: {\"product_name\": \"맥북\", \"rating\": 4.8, \"pros\": [\"성능\"], \"cons\": [], \"recommend\": true, \"sentiment_score\": 0.9, \"key_features_mentioned\": []}",
]

print("=== OutputFixingParser 테스트 ===")
fixing_success = 0
for i, bad_output in enumerate(malformed_outputs, 1):
    try:
        result = fixing_parser.parse(bad_output)
        fixing_success += 1
        print(f"[{i}] 수정 성공: {result.product_name}, rating={result.rating}")
    except Exception as e:
        print(f"[{i}] 수정 실패: {str(e)[:80]}")

print(f"\nOutputFixingParser 성공률: {fixing_success}/{len(malformed_outputs)}")

---
## 섹션 11. Langfuse로 성공률 추적

파싱 성공/실패 스코어를 Langfuse에 기록합니다.

In [ ]:
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
import uuid

# Langfuse 초기화
langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com"
)

langfuse_handler = CallbackHandler(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    session_id="ep08-json-function-calling"
)

print("Langfuse 초기화 완료")

In [ ]:
def tracked_parse(text: str, llm, schema, trace_name: str = "structured_parse") -> dict:
    """Langfuse 추적이 포함된 파싱 함수."""
    trace_id = str(uuid.uuid4())
    trace = langfuse.trace(
        id=trace_id,
        name=trace_name,
        input={"text_length": len(text), "schema": schema.__name__}
    )

    try:
        structured = llm.with_structured_output(schema)
        result = structured.invoke(
            f"다음 내용을 분석하세요:\n{text}",
            config={"callbacks": [langfuse_handler]}
        )

        # 성공 스코어 기록
        langfuse.score(
            trace_id=trace_id,
            name="parse_success",
            value=1,
            comment="Parsing succeeded"
        )
        trace.update(output={"success": True})
        return {"success": True, "result": result, "trace_id": trace_id}

    except Exception as e:
        # 실패 스코어 기록
        langfuse.score(
            trace_id=trace_id,
            name="parse_success",
            value=0,
            comment=f"Parsing failed: {str(e)[:100]}"
        )
        trace.update(output={"success": False, "error": str(e)})
        return {"success": False, "result": None, "trace_id": trace_id, "error": str(e)}


# 샘플 리뷰들로 추적 테스트
test_reviews = [
    SAMPLE_REVIEW,
    tricky_inputs[0],  # 아이러니
    tricky_inputs[1],  # 짧은 리뷰
]

print("=== Langfuse 추적 파싱 ===")
tracked_results = []
for i, review in enumerate(test_reviews, 1):
    outcome = tracked_parse(review, gpt_llm, ProductReview, f"review_parse_{i}")
    tracked_results.append(outcome)
    status = "성공" if outcome["success"] else "실패"
    print(f"[{i}] {status} | trace_id: {outcome['trace_id'][:8]}...")

# Langfuse 이벤트 플러시
langfuse.flush()
success_rate = sum(1 for r in tracked_results if r["success"]) / len(tracked_results)
print(f"\n전체 성공률: {success_rate:.0%}")
print("Langfuse 대시보드에서 스코어를 확인하세요: https://cloud.langfuse.com")

---
## 섹션 12. 실전 파이프라인: 대량 문서 → 구조화 데이터 추출 (배치 처리)

In [ ]:
# 뉴스 기사 샘플 배치 (10개)
sample_articles = [
    "삼성전자가 3분기 영업이익 9조1000억원을 기록했다. 반도체 부문이 흑자 전환하며 실적 개선을 이끌었다.",
    "정부가 2025년 최저임금을 10,030원으로 결정했다. 노동계는 인상률 부족을 이유로 반발했다.",
    "BTS 진이 전역 후 첫 솔로 앨범을 발표해 글로벌 팬들의 뜨거운 반응을 얻고 있다.",
    "네이버가 자체 개발 AI 하이퍼클로바X를 기업용 서비스로 출시했다. 초기 반응은 긍정적이다.",
    "한국 증시가 미국 금리 인상 우려로 3일 연속 하락했다. 코스피는 2,400선이 위협받고 있다.",
    "서울시가 2030년까지 대중교통 요금을 동결하겠다고 발표해 시민들의 환영을 받았다.",
    "현대차 아이오닉6가 유럽 올해의 차 시상식에서 3위에 올랐다. 전기차 경쟁력을 입증한 결과다.",
    "카카오페이 개인정보 유출 사태로 금융당국이 조사에 착수했다. 200만 명 정보가 노출됐다.",
    "2024 파리올림픽에서 한국 양궁이 금메달 5개로 역대 최다 기록을 세웠다.",
    "애플이 아이폰16 시리즈를 공개했다. AI 기능 강화가 핵심으로 삼성과의 경쟁이 심화된다.",
]

import time

def batch_parse_articles(articles: list, llm, batch_size: int = 3) -> list:
    """뉴스 기사를 배치로 파싱합니다."""
    structured = llm.with_structured_output(NewsArticle)
    results = []

    for i, article in enumerate(articles):
        try:
            result = structured.invoke(f"다음 기사를 분석하세요:\n{article}")
            results.append({"index": i+1, "success": True, "data": result.model_dump()})
        except Exception as e:
            results.append({"index": i+1, "success": False, "error": str(e)[:100]})

        # API 레이트 리밋 방지
        if (i + 1) % batch_size == 0:
            time.sleep(0.5)

    return results

print("배치 파싱 시작...")
start_time = time.time()
batch_results = batch_parse_articles(sample_articles, gpt_llm)
elapsed = time.time() - start_time

# 결과 집계
success_count = sum(1 for r in batch_results if r["success"])
print(f"\n배치 처리 완료")
print(f"성공: {success_count}/{len(sample_articles)} ({success_count/len(sample_articles):.0%})")
print(f"소요 시간: {elapsed:.1f}초")
print(f"기사당 평균: {elapsed/len(sample_articles):.1f}초")

# 카테고리 분포
categories = [r["data"]["category"] for r in batch_results if r["success"]]
from collections import Counter
print(f"\n카테고리 분포: {dict(Counter(categories))}")

In [ ]:
# 배치 결과 시각화
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

successful_results = [r for r in batch_results if r["success"]]
df_results = pd.DataFrame([r["data"] for r in successful_results])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 카테고리 분포
cat_counts = df_results["category"].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values,
            color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6'],
            alpha=0.85)
axes[0].set_title("Article Category Distribution", fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

# 감성 분포
sentiment_counts = df_results["sentiment"].value_counts()
colors_sentiment = {"긍정": "#2ecc71", "중립": "#95a5a6", "부정": "#e74c3c"}
bar_colors = [colors_sentiment.get(s, "#3498db") for s in sentiment_counts.index]
axes[1].bar(sentiment_counts.index, sentiment_counts.values,
            color=bar_colors, alpha=0.85)
axes[1].set_title("Sentiment Distribution", fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("batch_parsing_results.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n파싱된 데이터 (상위 3개):")
print(df_results[["title", "category", "sentiment"]].head(3).to_string(index=False))

---
## Exercise

### Exercise 1: 제품 리뷰 → Pydantic 모델로 파싱

**목표**: `ProductReview` 스키마에 새 필드를 추가하고, Claude vs GPT-4o mini 파싱 성능을 비교합니다.

In [ ]:
# TODO 1: ProductReview 스키마에 다음 필드를 추가하세요:
#   - purchase_intent: Literal["재구매", "고려", "미고려"] — 재구매 의도
#   - price_value: Literal["비쌈", "적당", "저렴"] — 가격 대비 가치 평가

# TODO 2: 한국어 제품 리뷰 20개를 작성하거나 인터넷에서 수집하세요 (다양한 제품, 다양한 길이)
reviews_to_test = [
    "여기에 리뷰를 추가하세요...",
    # 20개 리뷰 추가
]

# TODO 3: claude_llm과 gpt_llm 각각으로 20개 리뷰 파싱
# TODO 4: 성공률, 소요 시간, 평균 rating 값을 비교 테이블로 출력
# TODO 5: 실패한 케이스의 공통 패턴을 분석하고 1문단으로 작성

print("Exercise 1: TODO를 완성하세요!")

### Exercise 2: 재시도 전략 구현 + 성공률 측정

**목표**: 의도적으로 파싱하기 어려운 텍스트로 세 가지 재시도 전략의 성공률을 측정합니다.

In [ ]:
# TODO 1: tricky_inputs에 5개를 더 추가하세요 (총 10개)
# 아이디어: 극도로 짧은 리뷰, 숫자만 있는 리뷰, 전혀 관련 없는 텍스트 등

extended_tricky_inputs = tricky_inputs + [
    # 여기에 5개 추가
]

# TODO 2: 세 가지 전략으로 10개 입력 각각 파싱
#   전략 A: 기본 (재시도 없음) — parse_review_basic 사용
#   전략 B: @retry (max 3번) — safe_parse_with_retry 사용
#   전략 C: OutputFixingParser

# TODO 3: 결과를 다음 표 형식으로 채우세요:
strategy_comparison = pd.DataFrame({
    "전략": ["A: 기본", "B: @retry", "C: OutputFixingParser"],
    "성공률": ["?%", "?%", "?%"],
    "추가 비용 (토큰)": ["0", "?", "?"],
    "권장 상황": ["?", "?", "?"]
})
print(strategy_comparison.to_string(index=False))

# TODO 4: 어떤 전략을 언제 쓸 것인가? 결론 1문단 작성

print("\nExercise 2: TODO를 완성하세요!")